# H&M 거래 데이터 EDA 및 RFM 고객 세분화

EDA는 CSV 앞부분 10만 행으로 수행한다. 날짜순으로 정렬된 입력의 초기 거래를 보여 주는 표본이다. RFM은 원본 거래 전체를 대상으로 계산한다.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.pipeline import DataAnalyzer

analyzer = DataAnalyzer(root / "data" / "raw")
sample = analyzer.load_data(nrows=100_000)
print(sample.shape, sample["t_dat"].min().date(), sample["t_dat"].max().date())
sample.info()
print(sample.drop(columns=["customer_id", "article_id"]).head().to_string(index=False))
print(sample[["age", "price"]].describe().round(4).to_string())

## 결측치, 이상치 및 파생 변수

멤버십 상태별 연령 중앙값으로 대치한다. 그룹 중앙값이 없으면 전체 중앙값을 사용한다. IQR 범위를 벗어난 가격은 박스플롯에서 분포 차이를 확인하는 데 사용하며, RFM 집계에는 원래 가격을 사용한다.

In [ ]:
missing_before = sample["age"].isna().sum()
analyzer.handle_missing_values(strategy="group_median")
analyzer.add_text_features()
print(f"Missing age: {missing_before:,} -> {sample['age'].isna().sum():,}")

price_outliers = analyzer.detect_outliers("price")
price_inliers = sample.loc[~sample.index.isin(price_outliers.index), "price"]
print(f"Price outliers: {len(price_outliers):,} of {len(sample):,}")

numeric = sample[["age", "price", "word_count"]]
print(numeric.describe().round(4).to_string())
print(numeric.corr().round(4).to_string())

연령 결측치 584건(0.584%)이 대치 후 0건으로 줄었다. 가격 평균은 0.0296, 중앙값은 0.0254, 표준편차는 0.0224이며 1·3사분위수는 각각 0.0169와 0.0339였다. IQR 경계 밖에 7,895건(7.90%)이 있어 가격 분포는 중앙값과 사분위수도 함께 보는 편이 적절하다. 가격과 연령의 피어슨 상관계수는 0.0737, 가격과 상품명 단어 수는 -0.0970이다. 두 관계 모두 약해 이 표본에서 각 변수를 단독으로 가격 설명에 쓰기 어렵다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
sns.histplot(sample["age"], ax=axes[0])
axes[0].set(title="Age distribution", xlabel="Age (years)", ylabel="Transaction rows")
axes[1].boxplot([sample["price"], price_inliers])
axes[1].set(xticks=[1, 2], xticklabels=["All prices", "Within IQR bounds"],
            title="Price before and after IQR filtering", xlabel="Price selection", ylabel="Price (dataset units)")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
sns.heatmap(numeric.corr(), annot=True, vmin=-1, vmax=1, cmap="vlag", ax=axes[0])
axes[0].set(title="Pearson correlation", xlabel="Variable", ylabel="Variable")
plot_sample = sample.sample(2_000, random_state=42)
axes[1].scatter(plot_sample["age"], plot_sample["price"], alpha=0.25, s=10)
axes[1].set(title="Age and item price", xlabel="Age (years)", ylabel="Price (dataset units)")
plt.show()

## 이미지 피처

`data/raw/images/`에 원본 이미지가 있을 때 거래 표본에 등장한 상품 최대 100개의 픽셀 평균과 표준편차를 계산한다. 각 이미지는 Matplotlib으로 배열로 읽고 NumPy로 통계량을 계산한다. 이미지의 원본 픽셀 스케일을 유지한다.

In [ ]:
image_dir = analyzer.data_path / "images"
if image_dir.is_dir():
    features = []
    for article_id in sample["article_id"].dropna().drop_duplicates():
        path = image_dir / article_id[:3] / f"{article_id}.jpg"
        if path.is_file():
            pixel_mean, pixel_std = analyzer.image_statistics(plt.imread(path))
            features.append((article_id, pixel_mean, pixel_std))
        if len(features) == 100:
            break
    image_features = pd.DataFrame(features, columns=["article_id", "image_mean", "image_std"])
    sample = sample.merge(image_features, on="article_id", how="left", validate="many_to_one")
    print(f"Matched images: {len(image_features)}")
else:
    print("Image directory missing: data/raw/images/")

## RFM 및 고객 세그먼트

Recency는 최종 거래일부터 마지막 구매일까지의 일수, Frequency는 거래 행 수, Monetary는 원본 가격 합계다. 동점 고객에는 동일한 백분위 순위를 적용한다. 세그먼트 규칙은 VIP → Loyal → New → At Risk → Regular 순으로 적용한다. New는 최근 구매했고 관측 기간 내 구매 행이 1건인 고객이다.

In [ ]:
rfm = analyzer.calculate_rfm(chunksize=2_000_000)
summary = rfm.groupby("segment").agg(
    customers=("frequency", "size"),
    median_recency=("recency", "median"),
    median_frequency=("frequency", "median"),
    total_monetary=("monetary", "sum"),
)
summary["customer_pct"] = summary["customers"] / len(rfm) * 100
summary["monetary_pct"] = summary["total_monetary"] / rfm["monetary"].sum() * 100

print(f"Reference date: {rfm.attrs['reference_date'].date()}")
print(f"Customers: {len(rfm):,}, transaction rows: {rfm['frequency'].sum():,}")
print(summary.round(2).to_string())

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
summary["customers"].sort_values().plot.barh(ax=ax)
ax.set(title="Customers by RFM segment", xlabel="Customers", ylabel="Segment")
plt.show()

VIP의 구매 행 수 중앙값은 43건, At Risk는 3건이다. 두 집단의 Recency 중앙값도 각각 29일과 483일로 차이가 크므로, VIP 유지와 At Risk 재활성화는 별도 캠페인으로 검증할 가치가 있다.

## 월별 거래량

원본 거래의 날짜 열만 나눠 읽어 월별 거래 행 수를 집계한다. 처음과 마지막 달에는 월 전체가 포함되지 않는다.

In [ ]:
monthly_counts = []
for dates in pd.read_csv(analyzer.data_path / "transactions_train.csv", usecols=["t_dat"],
                         parse_dates=["t_dat"], chunksize=2_000_000):
    monthly_counts.append(dates["t_dat"].dt.to_period("M").value_counts())
monthly = pd.concat(monthly_counts).groupby(level=0).sum().sort_index()
assert monthly.sum() == rfm["frequency"].sum()

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(monthly.index.to_timestamp(), monthly.to_numpy(), marker="o")
ax.set(title="Monthly transaction rows", xlabel="Month", ylabel="Transaction rows")
plt.show()
print(f"Highest monthly count: {monthly.idxmax()} ({monthly.max():,})")

거래 행 수는 2019년 6월 1,906,202건으로 가장 많았다. 계절성을 비교할 때는 월 전체가 포함된 달을 사용하고, 상품 판매량과 실제 주문 수는 별도 주문 데이터로 확인한다.